# VRCaps Colab

**GitHub:** https://github.com/rckarakurt/g3d

Turntable: **mesh Y ekseninde doner**, kamera sabit (+Z).
- **0** = duz yuz | **-90 / +90** = tam yan profil

Sifirdan: Restart -> **0 -> 1 -> 1b -> 2 -> 5b -> 3 -> 4 -> 6 -> 7**



## 0. GitHub clone + temiz baslangic

**Repo:** https://github.com/rckarakurt/g3d

### Sifirdan Colab

1. **Runtime → Restart session**
2. **Bolum 0** calistir (`CLEAN_OLD_OUTPUTS = True` varsayilan)
3. Sirayla: **1 → 1b → 2 → 5b → 3 → 4 → 6 → 7**

### Aci konvansiyonu (view bank)

| Bank acisi | Gorunum |
|------------|---------|
| **0°** | Duz yuz bize bakan (+Z) |
| **-90°** | Sol yan |
| **+90°** | Sag yan |

- Sadece **Y ekseni** (dikey), **-90 .. +90**
- Unity `view_plane_deg` (0..180) otomatik `-90` offset ile eslestirilir

### Kod guncelleme

GitHub push sonrasi Bolum 0 tekrar calistir. Eski notebook kullanma — GitHub'daki `.ipynb` indir.




In [ ]:
# 0 — GitHub'dan VRCaps reposunu klonla / guncelle
import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np

# ============ AYARLAR ============
REPO_URL = "https://github.com/rckarakurt/g3d.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/g3d")

# Sifirdan baslarken True yap — eski view_bank / ciktilari siler
CLEAN_OLD_OUTPUTS = True
# ================================


def _run(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess:
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode, cmd, result.stdout, result.stderr
        )
    return result


def _clone() -> None:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    _run(
        [
            "git",
            "clone",
            "-b",
            REPO_BRANCH,
            "--depth",
            "1",
            REPO_URL,
            str(REPO_DIR),
        ]
    )
    print("Repo klonlandi:", REPO_DIR)


def _update() -> None:
    """Shallow clone icin pull yerine fetch + reset (--ff-only Colab'da sik kirilir)."""
    try:
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH, "--depth", "1"])
        _run(["git", "-C", str(REPO_DIR), "checkout", "-f", REPO_BRANCH])
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"])
        print("Repo guncellendi:", REPO_DIR)
    except subprocess.CalledProcessError:
        print("fetch/reset basarisiz — yeniden klonlaniyor...")
        _clone()


if REPO_DIR.exists() and (REPO_DIR / ".git").is_dir():
    _update()
else:
    _clone()

os.environ["VRCAPS_REPO"] = str(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

from vrcaps_colab_bootstrap import bootstrap

orbit, gaze = bootstrap()
if not (orbit / "coating_utils.py").exists():
    raise FileNotFoundError(
        f"Scriptler bulunamadi: {orbit}\n"
        "REPO_URL ve REPO_BRANCH dogru mu? Once GitHub'a push edin."
    )

from colab_content_paths import print_repo_info

if CLEAN_OLD_OUTPUTS:
    import shutil as _shutil

    for _p in (
        Path("/content/ply_styleshot_out"),
        Path("/content/gaze_composite"),
        Path("/content/medical_gan_dataset"),
    ):
        if _p.exists():
            _shutil.rmtree(_p)
            print("Silindi:", _p)

_commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
    capture_output=True,
    text=True,
    check=False,
)
if _commit.stdout.strip():
    print("Git commit:", _commit.stdout.strip())

from turntable_render import lumen_camera_c2w, rotate_vertices_y, wall_azimuth_grid

print("Turntable: mesh Y ekseninde doner, kamera sabit (+Z, 0=duz yuz)")
print("Ornek grid:", wall_azimuth_grid(45, 90))
_t = np.zeros(3)
_c2w = lumen_camera_c2w(1.0, _t, obliquity_deg=18.0)
print("Sabit kamera eye:", _c2w[:3, 3])
for _az in (-90, 0, 90):
    _v = rotate_vertices_y(np.array([[0.5, 0, 0.2], [-0.5, 0, 0.2]]), -float(_az), _t)
    print(f"  az {_az:+4d}: test nokta donduruldu -> {_v[0]}")

print_repo_info()
print("\nSonraki adim: 1 -> 1b -> 2 -> 5b -> 3 -> 4 -> 6 -> 7")



## 1. Kurulum




In [ ]:
# PLY + StyleShot + xatlas — Colab kurulum
import os
import pathlib
import site
from pathlib import Path

!pip install -q "numpy==2.0.2" "scipy==1.14.1"
!pip install -q "diffusers==0.32.2" "transformers==4.46.3" "accelerate==0.34.2" "einops==0.7.0"
!pip install -q opencv-python-headless huggingface_hub safetensors matplotlib tqdm pillow gdown certifi
!pip install -q "open3d>=0.18.0" xatlas trimesh

# Colab SSL duzeltmesi (CERTIFICATE_VERIFY_FAILED)
import os
import ssl

import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl._create_unverified_context
print("SSL patch OK")

os.chdir("/content")
if not Path("StyleShot").exists():
    !git clone -q https://github.com/open-mmlab/StyleShot.git
os.chdir("/content/StyleShot")

!pip install -q addict future lmdb pyyaml requests yapf
!pip install -q basicsr --no-deps 2>/dev/null || pip install -q git+https://github.com/XPixelGroup/BasicSR.git --no-deps

IP = Path("ip_adapter/ip_adapter.py")
t = IP.read_text(encoding="utf-8")
t = t.replace(
    "mult = len(controlnet.nets) if isinstance(controlnet, MultiControlNetModel) else 1",
    "mult = len(controlnet.nets) if hasattr(controlnet, 'nets') else 1",
)
t = t.replace(
    "if isinstance(controlnet, MultiControlNetModel) and isinstance(controlnet_conditioning_scale, float):",
    "if hasattr(controlnet, 'nets') and isinstance(controlnet_conditioning_scale, float):",
)
t = t.replace(
    "controlnet_keep.append(keeps[0] if isinstance(controlnet, ControlNetModel) else keeps)",
    "controlnet_keep.append(keeps[0] if not hasattr(controlnet, 'nets') else keeps)",
)
t = t.replace(
    "if isinstance(self.pipe.controlnet, MultiControlNetModel):",
    "if hasattr(self.pipe.controlnet, 'nets'):",
)
OLD_GP = """        global_pool_conditions = (
            controlnet.config.global_pool_conditions
            if isinstance(controlnet, ControlNetModel)
            else controlnet.nets[0].config.global_pool_conditions
        )"""
NEW_GP = """        if hasattr(controlnet, 'nets'):
            global_pool_conditions = controlnet.nets[0].config.global_pool_conditions
        else:
            global_pool_conditions = getattr(controlnet.config, 'global_pool_conditions', False)"""
t = t.replace(OLD_GP, NEW_GP)
t = t.replace(
    "        if isinstance(controlnet, ControlNetModel):\n            image = self.prepare_image(",
    "        if not hasattr(controlnet, 'nets'):\n            image = self.prepare_image(",
)
t = t.replace(
    "        elif isinstance(controlnet, MultiControlNetModel):\n            images = []",
    "        else:\n            images = []",
)
t = t.replace("        else:\n            assert False\n", "")
IP.write_text(t, encoding="utf-8")

for sp in site.getsitepackages():
    bsr = pathlib.Path(sp) / "basicsr"
    if not bsr.exists():
        continue
    for f in bsr.rglob("*.py"):
        txt = f.read_text(encoding="utf-8", errors="ignore")
        new = txt.replace(
            "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
            "from torchvision.transforms.functional import rgb_to_grayscale",
        )
        if new != txt:
            f.write_text(new, encoding="utf-8")

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

import sys
sys.path.insert(0, "/content/g3d")
try:
    from vrcaps_colab_bootstrap import bootstrap
    orbit, _ = bootstrap()
    print("Script kaynagi (GitHub):", orbit)
except ImportError:
    print("GitHub bootstrap yok — once Bolum 0 calistirin")




## 1b. Modeller




In [ ]:
# 1b — StyleShot HF modelleri (ip.bin + SD1.5 + CLIP, ~8 GB)
import os
import sys
from pathlib import Path

sys.path.insert(0, "/content/g3d")
from vrcaps_colab_bootstrap import bootstrap

bootstrap()
from colab_content_paths import install_sys_path

install_sys_path()

os.chdir("/content/StyleShot")

from styleshot_models import ensure_styleshot_models, ip_bin_path

ensure_styleshot_models()
print("ip.bin:", ip_bin_path())




## 2. PLY




In [ ]:
# 2 — Drive paylasim klasorunden polyp PLY indir
import sys
from pathlib import Path

sys.path.insert(0, "/content/g3d")
from vrcaps_colab_bootstrap import bootstrap

bootstrap()
from colab_content_paths import install_sys_path

install_sys_path()

from colab_net import configure_colab_ssl

configure_colab_ssl()

from drive_ply_fetch import (
    POLYP_DRIVE_FOLDER_URL,
    fetch_ply_from_drive_folder,
    load_selected_ply,
)

POLYP_NAME = "polyp_0004.ply"
FORCE_REDOWNLOAD = False  # eski indirme varsa bir kez True yapin

existing = load_selected_ply()
if existing and not FORCE_REDOWNLOAD:
    PLY_PATH = existing
    print("Onceki indirme kullaniliyor:", PLY_PATH)
else:
    PLY_PATH = fetch_ply_from_drive_folder(
        POLYP_NAME,
        force_download=FORCE_REDOWNLOAD,
    )

print("Kaynak klasor:", POLYP_DRIVE_FOLDER_URL)
print("PLY hazir:", PLY_PATH)



## 5. Unity dataset — Drive yolu

### Drive (kaynak)

```
MyDrive/vrcaps/medical_gan_dataset/
├── camera.json
├── rgb/
├── depth/
└── poses/
```

Colab tam yol:
```
/content/drive/MyDrive/vrcaps/medical_gan_dataset
```

### Calisma kopyasi (Colab oturumu)

Bolum 5b bu klasoru **`/content/medical_gan_dataset/`** altina kopyalar.

### Bolum 5b

```python
SOURCE = "drive_folder"   # varsayilan
```

1. `drive.mount()`
2. `vrcaps/medical_gan_dataset` klasorunu kopyala
3. Klasor yoksa `vrcaps/unity-vr-caps-dataset.rar` dener

### Diger /content yollari

| Cikti | Yol |
|-------|-----|
| Polyp render | `/content/ply_styleshot_out/view_bank/` |
| Composite | `/content/gaze_composite/` |




## 5b. Dataset




In [ ]:
# 05 — Unity dataset: MyDrive/vrcaps/medical_gan_dataset → /content
from __future__ import annotations

import sys
from pathlib import Path

import sys
from pathlib import Path

sys.path.insert(0, "/content/g3d")
from vrcaps_colab_bootstrap import bootstrap

bootstrap()
from colab_content_paths import install_sys_path

install_sys_path()
from colab_content_paths import DATASET_DRIVE, UNITY_DATASET, sync_dataset_from_drive

# ============ AYARLAR ============
# Drive yolu (Colab tam yol):
#   /content/drive/MyDrive/vrcaps/medical_gan_dataset
#
# Sizin Drive yolu:
#   MyDrive/vrcaps/medical_gan_dataset
#
SOURCE = "drive_folder"  # "drive_folder" | "drive_rar" | "archive"

# RAR yedek (klasor yoksa dene) — MyDrive/vrcaps/ altinda
DRIVE_RAR_NAME = "unity-vr-caps-dataset.rar"  # veya medical_gan_dataset.rar

ARCHIVE_PATH = None  # SOURCE="archive" ise /content/...zip
EXTRACT_TO = UNITY_DATASET
# ==================================


def _load_rar_from_drive() -> None:
    import shutil
    import subprocess
    import zipfile
    from colab_content_paths import DRIVE_ROOT

    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    for name in (DRIVE_RAR_NAME, "medical_gan_dataset.rar", "unity-vr-caps-dataset.rar"):
        rar_path = DRIVE_ROOT / name
        if not rar_path.exists():
            continue
        print("Drive RAR:", rar_path)
        local = Path("/content") / name
        if not local.exists():
            shutil.copy2(rar_path, local)
        staging = Path("/content/_dataset_unpack")
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir()
        subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=False)
        subprocess.run(["unrar", "x", "-o+", str(local), str(staging) + "/"], check=True)
        # finalize inline
        dest = EXTRACT_TO
        if dest.exists():
            shutil.rmtree(dest)
        if (staging / "medical_gan_dataset" / "rgb").exists():
            shutil.move(str(staging / "medical_gan_dataset"), str(dest))
        elif len(list(staging.iterdir())) == 1 and (next(staging.iterdir()) / "rgb").exists():
            shutil.move(str(next(staging.iterdir())), str(dest))
        elif (staging / "rgb").exists():
            shutil.move(str(staging), str(dest))
        shutil.rmtree(staging, ignore_errors=True)
        return
    raise FileNotFoundError(
        f"Drive'da ne klasor ne RAR bulundu.\n"
        f"  Klasor: {DATASET_DRIVE}\n"
        f"  RAR:    {DRIVE_ROOT / DRIVE_RAR_NAME}"
    )


print("Drive kaynak:", DATASET_DRIVE)
print("Hedef:       ", EXTRACT_TO)

if SOURCE == "drive_folder":
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    if (DATASET_DRIVE / "rgb").is_dir():
        sync_dataset_from_drive(EXTRACT_TO)
    else:
        print("Klasor yok, RAR deneniyor...")
        _load_rar_from_drive()
elif SOURCE == "drive_rar":
    _load_rar_from_drive()
else:
    raise ValueError("archive modu icin eski script veya ARCHIVE_PATH kullanin")

n_rgb = len(list((EXTRACT_TO / "rgb").glob("*.png")))
if n_rgb == 0:
    raise RuntimeError(f"rgb/ bos — kontrol edin: {EXTRACT_TO}")
print(f"\nHazir: {EXTRACT_TO}  ({n_rgb} kare)")
print("Sonraki: Bolum 3 → 6 → 7")



## 3. View bank




In [ ]:
# 12 — Drive PLY: xatlas UV + StyleShot texture + turntable render
import json
import shutil
import sys
from pathlib import Path

# ============ AYARLAR ============
POLYP_NAME = "polyp_0004.ply"

OUT_ROOT = Path("/content/ply_styleshot_out")
COPY_TO_DRIVE = False  # True = istege bagli Drive yedegi

ATLAS_TEX_SIZE = 2048
TEXTURE_SEED = 42
CONTROLNET_SCALE = 0.55
USE_CONTENT_ENCODER = False

PROMPT = (
    "colonoscopy image of a sessile polyp on pink mucosa, "
    "smooth organic surface, realistic endoscopic texture, clinical photography"
)

AZIMUTH_BIN_DEG = 5          # bank aci adimi (None = otomatik)
AZIMUTH_HALF_SPAN_DEG = 90   # -90 .. +90 (0 = duz yuz, Y ekseni)
SYNC_AZIMUTHS_FROM_UNITY = True
# Oncelik: /content/medical_gan_dataset (Bolum 5b); yoksa Drive fallback
UNITY_DATASET = Path("/content/medical_gan_dataset")

OBLIQUITY_DEG = 18.0  # on gorunumde hafif Y egimi; ±90 yan profilde etkisiz
RENDER_SIZE = 768
WALL_MOUNTED = True
DISTANCE_SCALE = 2.4  # mesh etrafinda donus icin kamera mesafesi
# ================================

import os

os.environ["VRCAPS_USE_KVASIR"] = "0"  # Colab SSL: Kvasir indirme kapali, sentetik ref

os.chdir("/content/StyleShot")
sys.path.insert(0, "/content/g3d")
from vrcaps_colab_bootstrap import bootstrap

bootstrap()
from colab_content_paths import install_sys_path

orbit, _ = install_sys_path()
sys.path.insert(0, "/content/StyleShot")

from drive_ply_fetch import fetch_ply_from_drive_folder, load_selected_ply
from ply_loader import center_mesh, load_mesh_ply, mesh_bounds_radius
from mesh_uv import save_uv_preview, unwrap_mesh_uv
from style_encoder_drive import ensure_style_encoder
from turntable_render import wall_azimuth_grid
from unity_view_bank_angles import azimuths_from_gaze_dataset, save_angle_meta
from colab_content_paths import PLY_OUT_DRIVE, copy_tree_if_requested, resolve_unity_dataset
import styleshot_texture as _styleshot_tex


def _patch_style_ref_no_kvasir() -> None:
    """Eski gomulu script olsa bile Kvasir ag indirmesini atla."""
    import cv2
    import numpy as np
    import shutil

    cache = _styleshot_tex.STYLE_REF_CACHE

    def ensure_style_ref(dest=None):
        dest = Path(dest or cache)
        if dest.exists() and dest.stat().st_size > 10_000:
            return dest
        for candidate in (
            Path("/content/drive/MyDrive/vrcaps/kvasir_style_ref.jpg"),
            Path("/content/drive/MyDrive/vrcaps/kvasir_style_ref.png"),
        ):
            if candidate.exists():
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(candidate, dest)
                return dest
        if hasattr(_styleshot_tex, "_synthetic_mucosa_style_ref"):
            return _styleshot_tex._synthetic_mucosa_style_ref(dest)
        dest.parent.mkdir(parents=True, exist_ok=True)
        rng = np.random.default_rng(42)
        base = np.array([188, 108, 118], dtype=np.float32)
        img = base + rng.normal(0, 22, (512, 512, 3))
        yy, xx = np.mgrid[0:512, 0:512]
        vignette = 1.0 - 0.35 * np.sqrt(((xx - 256) / 256) ** 2 + ((yy - 256) / 256) ** 2)
        img = np.clip(img * vignette[..., None], 0, 255).astype(np.uint8)
        cv2.imwrite(str(dest), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        print("Sentetik stil referansi (Kvasir atlandi):", dest)
        return dest

    _styleshot_tex.ensure_style_ref = ensure_style_ref
    print("Kvasir indirme devre disi — sentetik stil ref kullanilacak")


_patch_style_ref_no_kvasir()
from styleshot_texture import generate_uv_texture
from turntable_render import render_turntable_views

import cv2
import numpy as np

PLY_PATH = load_selected_ply()
if PLY_PATH is None:
    print("Bolum 2 calistirilmamis — PLY simdi indiriliyor...")
    PLY_PATH = fetch_ply_from_drive_folder(POLYP_NAME)

print("Style encoder indiriliyor / kontrol...")
style_encoder_path = ensure_style_encoder()
print("Encoder:", style_encoder_path)

OUT_ROOT.mkdir(parents=True, exist_ok=True)
unity_dataset = resolve_unity_dataset()
uv_dir = OUT_ROOT / "mesh_uv"
tex_dir = OUT_ROOT / "uv_texture"
view_dir = OUT_ROOT / "view_bank"

print("1/4 PLY yukleniyor:", PLY_PATH)
verts, faces, _normals, colors = load_mesh_ply(PLY_PATH)
verts, _center = center_mesh(verts)
radius_mm = mesh_bounds_radius(verts)
print(f"   {len(verts)} vert, {len(faces)} tri, radius ~{radius_mm:.2f} mm")

print("2/4 xatlas UV unwrap...")
verts_uv, faces_uv, uvs = unwrap_mesh_uv(verts, faces, atlas_size=ATLAS_TEX_SIZE)
uv_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    uv_dir / "polyp_uv.npz",
    vertices=verts_uv,
    faces=faces_uv,
    uvs=uvs,
    tex_size=ATLAS_TEX_SIZE,
    source_ply=str(PLY_PATH),
)
blank = np.full((ATLAS_TEX_SIZE, ATLAS_TEX_SIZE, 3), 180, dtype=np.uint8)
save_uv_preview(blank, uv_dir / "uv_layout_preview.png")
print("   ->", uv_dir / "polyp_uv.npz")

print("3/4 StyleShot texture uretiliyor...")
texture, tex_meta = generate_uv_texture(
    verts,
    faces,
    colors,
    out_size=ATLAS_TEX_SIZE,
    seed=TEXTURE_SEED,
    prompt=PROMPT,
    controlnet_scale=CONTROLNET_SCALE,
    use_content_encoder=USE_CONTENT_ENCODER,
    elevation_deg=OBLIQUITY_DEG,
)
tex_dir.mkdir(parents=True, exist_ok=True)
tex_path = tex_dir / "polyp_uv_texture.png"
cv2.imwrite(str(tex_path), cv2.cvtColor(texture, cv2.COLOR_RGB2BGR))
save_uv_preview(texture, tex_dir / "polyp_uv_texture_preview.png")
(tex_dir / "texture_meta.json").write_text(json.dumps(tex_meta, indent=2), encoding="utf-8")
print("   ->", tex_path)

if SYNC_AZIMUTHS_FROM_UNITY and unity_dataset.exists() and (unity_dataset / "rgb").exists():
    gaze_csv = unity_dataset / "poses" / "gaze_views.csv"
    if not gaze_csv.exists() and (unity_dataset / "rgb").exists():
        from export_gaze_views import export_gaze_views

        print("gaze_views.csv yok — Unity dataset uzerinde uretiliyor...")
        export_gaze_views(unity_dataset, write_plot=False)
    AZIMUTHS_DEG, angle_meta = azimuths_from_gaze_dataset(
        unity_dataset,
        bin_deg=AZIMUTH_BIN_DEG,
        half_span_deg=AZIMUTH_HALF_SPAN_DEG,
        fill_span=True,
    )
    save_angle_meta(angle_meta, view_dir / "unity_angle_sync.json")
    print(
        f"Unity aci senkron: {len(AZIMUTHS_DEG)} gorunum, "
        f"bank span {angle_meta['bank_az_min_deg']:.1f}.."
        f"{angle_meta['bank_az_max_deg']:.1f}° (0=duz yuz), bin={angle_meta['bin_deg']}°"
    )
else:
    step = int(AZIMUTH_BIN_DEG or 5)
    AZIMUTHS_DEG = wall_azimuth_grid(step, int(AZIMUTH_HALF_SPAN_DEG))
    print(
        f"Unity dataset yok — sabit grid -{AZIMUTH_HALF_SPAN_DEG}..+{AZIMUTH_HALF_SPAN_DEG} "
        f"step={step}° ({len(AZIMUTHS_DEG)} gorunum)"
    )

print(f"4/4 Duvar-yapisik render ({len(AZIMUTHS_DEG)} aci): ilk/son = {AZIMUTHS_DEG[0]:.0f}..{AZIMUTHS_DEG[-1]:.0f}")
manifest = render_turntable_views(
    verts_uv,
    faces_uv,
    uvs,
    texture,
    AZIMUTHS_DEG,
    view_dir,
    width=RENDER_SIZE,
    height=RENDER_SIZE,
    elevation_deg=OBLIQUITY_DEG,
    distance_scale=DISTANCE_SCALE,
    wall_mounted=WALL_MOUNTED,
    orbit_mode="lumen",  # Y ekseni -90..+90, 0=duz yuz (+Z)
)
print("   ->", view_dir)
for entry in manifest["views"]:
    print(f"      az={entry['azimuth_deg']:5.0f}  {entry['file']}")

print("5/5 Onizleme icin Bolum 4 calistirin (15° grid strip).")

meta = {
    "ply": str(PLY_PATH),
    "style_encoder": str(style_encoder_path),
    "texture_meta": tex_meta,
    "atlas_tex_size": ATLAS_TEX_SIZE,
    "azimuths_deg": AZIMUTHS_DEG,
    "azimuth_bin_deg": AZIMUTH_BIN_DEG,
    "azimuth_half_span_deg": AZIMUTH_HALF_SPAN_DEG,
    "sync_azimuths_from_unity": SYNC_AZIMUTHS_FROM_UNITY,
    "unity_dataset": str(unity_dataset),
    "obliquity_deg": OBLIQUITY_DEG,
    "wall_mounted": WALL_MOUNTED,
    "mesh_radius_mm": radius_mm,
}
(OUT_ROOT / "pipeline_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("Cikti (content):", OUT_ROOT)
print("  view_bank:", view_dir)
copy_tree_if_requested(OUT_ROOT, PLY_OUT_DRIVE, enabled=COPY_TO_DRIVE)

print("Tamamlandi.")




## 4. Onizleme




In [ ]:
# 13 — Full 360° azimuth preview (0..360, arka gorunumler dahil)
import json
import sys
from pathlib import Path

import cv2
import numpy as np

# ============ AYARLAR ============
OUT_ROOT = Path("/content/ply_styleshot_out")
PREVIEW_DIR = OUT_ROOT / "view_preview_full360"

PREVIEW_STEP_DEG = 15
PREVIEW_HALF_SPAN_DEG = 90   # -90 .. +90
ORBIT_MODE = "lumen"

PREVIEW_RENDER_SIZE = 512
PREVIEW_THUMB_HEIGHT = 180
PREVIEW_GRID_COLS = 8
# ================================

import sys
from pathlib import Path

sys.path.insert(0, "/content/g3d")
from vrcaps_colab_bootstrap import bootstrap

bootstrap()
from colab_content_paths import install_sys_path

install_sys_path()
from turntable_render import (
    preview_azimuth_grid,
    render_turntable_views,
    show_view_strip_colab,
)

uv_npz = OUT_ROOT / "mesh_uv" / "polyp_uv.npz"
tex_path = OUT_ROOT / "uv_texture" / "polyp_uv_texture.png"
meta_path = OUT_ROOT / "pipeline_meta.json"

if not uv_npz.exists() or not tex_path.exists():
    raise FileNotFoundError(
        f"UV/texture yok.\n  {uv_npz}\n  {tex_path}\nOnce Bolum 3 (pipeline) calistirin."
    )

obliquity_deg = 18.0
wall_mounted = True
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    obliquity_deg = float(meta.get("obliquity_deg", obliquity_deg))
    wall_mounted = bool(meta.get("wall_mounted", wall_mounted))

pack = np.load(uv_npz)
verts = pack["vertices"]
faces = pack["faces"]
uvs = pack["uvs"]
texture = cv2.cvtColor(cv2.imread(str(tex_path)), cv2.COLOR_BGR2RGB)

azimuths = preview_azimuth_grid(PREVIEW_STEP_DEG, PREVIEW_HALF_SPAN_DEG)
print(f"Onizleme: {ORBIT_MODE} orbit, {len(azimuths)} aci ({azimuths[0]:.0f}..{azimuths[-1]:.0f})")
print("  0=duz yuz (+Z), -90=sol yan, +90=sag yan  (Y ekseni, duvara yapisik)")

manifest = render_turntable_views(
    verts,
    faces,
    uvs,
    texture,
    azimuths,
    PREVIEW_DIR,
    width=PREVIEW_RENDER_SIZE,
    height=PREVIEW_RENDER_SIZE,
    elevation_deg=obliquity_deg,
    distance_scale=2.8,
    wall_mounted=wall_mounted,
    orbit_mode=ORBIT_MODE,
)
print("Manifest:", PREVIEW_DIR / "view_manifest.json")

strip = show_view_strip_colab(
    PREVIEW_DIR,
    thumb_height=PREVIEW_THUMB_HEIGHT,
    max_cols=PREVIEW_GRID_COLS,
    save=True,
    strip_name="view_strip_full360.png",
)
print("Kaydedildi:", PREVIEW_DIR / "view_strip_full360.png")



## 6. Gaze




In [ ]:
# 14 — Gaze anchor + viewing angles + trajectory map
from __future__ import annotations

import json
import sys
from pathlib import Path

get_ipython().run_line_magic("pip", "install -q pandas scipy matplotlib")

# ============ AYARLAR ============
COPY_TO_DRIVE = False  # True = istege bagli Drive yedegi

MARKED_RGB = True
MARKED_MAX_FRAMES = None
# ================================

sys.path.insert(0, "/content/g3d")
from vrcaps_colab_bootstrap import bootstrap

bootstrap()
from colab_content_paths import install_sys_path

install_sys_path()

from colab_content_paths import (
    GAZE_OUT_DRIVE,
    copy_tree_if_requested,
    resolve_unity_dataset,
)
from export_gaze_trajectory_map import export_gaze_trajectory_map
from export_gaze_views import export_gaze_views

if MARKED_RGB:
    from export_gaze_rgb_marked import export_marked_rgb


def _require_dataset(root: Path) -> None:
    missing = []
    for rel in ("camera.json", "rgb", "depth"):
        if not (root / rel).exists():
            missing.append(str(root / rel))
    pose_candidates = [
        root / "poses" / "poses_with_directions.csv",
        root / "poses" / "poses_per_frame.csv",
        root / "poses" / "position_rotation.csv",
    ]
    if not any(p.exists() for p in pose_candidates):
        missing.append(str(root / "poses/*.csv"))
    if missing:
        raise FileNotFoundError(
            "Dataset eksik:\n  "
            + "\n  ".join(missing)
            + "\n\nOnce Bolum 5b calistirin (Drive -> /content/medical_gan_dataset)."
        )
    n_rgb = len(list((root / "rgb").glob("*.png")))
    n_depth = len(list((root / "depth").glob("*.png")))
    print(f"Dataset OK: {root}")
    print(f"  rgb={n_rgb}  depth={n_depth}")
    cam = json.loads((root / "camera.json").read_text(encoding="utf-8"))
    print(f"  image {cam['image_width']}x{cam['image_height']}  depth_scale={cam['depth_scale']}")


def _show_colab_previews(dataset: Path) -> None:
    from IPython.display import Image, display

    traj = dataset / "trajectory"
    for name in ("gaze_trajectory_map.png", "gaze_analysis.png", "trajectory_with_directions.png"):
        path = traj / name
        if path.exists():
            print(name)
            display(Image(filename=str(path)))
    csv_path = dataset / "poses" / "gaze_views.csv"
    if csv_path.exists():
        import pandas as pd

        df = pd.read_csv(csv_path)
        print("gaze_views.csv (ilk 5 satir):")
        display(
            df[
                [
                    "frame",
                    "view_plane_deg",
                    "view_azimuth_deg",
                    "view_elevation_deg",
                    "is_gazing",
                    "distance_m",
                ]
            ].head()
        )
        vp = df["view_plane_deg"].astype(float)
        print(
            f"view_plane_deg: {vp.min():.1f} .. {vp.max():.1f}  "
            f"(n={len(df)}, gazing={int(df['is_gazing'].sum())})"
        )


dataset = resolve_unity_dataset()
_require_dataset(dataset)

print("\n1/3 Gaze anchor + acilar...")
meta = export_gaze_views(dataset, write_plot=True)
print("  anchor:", meta["anchor_pos"])
print("  orbit span:", meta.get("orbit_theta_span_deg"))
print("  gazing:", meta.get("gazing_frame_count"), "/", meta.get("frame_count"))

print("\n2/3 Trajectory map (acili)...")
map_path = export_gaze_trajectory_map(dataset)
print("  ->", map_path)

if MARKED_RGB:
    print("\n3/3 RGB overlay (crosshair + view_plane_deg)...")
    out_rgb = dataset / "gaze_rgb" / "rgb"
    export_marked_rgb(dataset, out_rgb, gazing_only=False, clean=True)
    print("  ->", out_rgb)

copy_tree_if_requested(dataset, GAZE_OUT_DRIVE, enabled=COPY_TO_DRIVE)

print("\n=== Tamamlandi ===")
print("Cikti:", dataset)
_show_colab_previews(dataset)



## 7. Aci eslestirmeli composite (Unity + mesh)

Her Unity karesine, **aynı bakis acisindaki** StyleShot mesh render yapilir.

### Mantik

```
Unity kare N
  view_plane_deg = 42.3°   (gaze_views.csv)
       ↓
view_bank: view_az040.png + view_az045.png  (blend)
       ↓
ekran merkezine (anchor) yapistir → composite
```

### Gerekli

| Kaynak | Yol |
|--------|-----|
| Unity RGB + gaze | `/content/medical_gan_dataset/` |
| Mesh view bank | `/content/ply_styleshot_out/view_bank/` |
| gaze_views.csv | Bolum 6 veya otomatik |

### Cikti

```
/content/gaze_composite/
├── rgb/000000.png ...      ← polipli Unity kareleri
├── debug/match_*.png       ← Unity | mesh | composite (aci etiketli)
├── trajectory_composite.mp4
└── composite_manifest.csv  ← frame, view_plane_deg, bank_azimuth
```

### Calistirma

Once: **5b** (Drive dataset) → **3** (view bank, Unity acilariyla) → **6** (gaze) → **7**

Bolum 7 ayarlari:
- `STRICT_ANGLE_MATCH = True` — kare bazli aci eslestirme
- `WRITE_DEBUG = True` — aci kontrolu icin uc lu onizleme




## 7. Composite




In [ ]:
# 15 — Aci eslestirmeli composite: Unity RGB + mesh (view_plane_deg)
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

get_ipython().run_line_magic("pip", "install -q opencv-python-headless scipy")

# ============ AYARLAR ============
# Her Unity karesi icin:
#   view_plane_deg (gaze_views.csv) -> view_bank'teki en yakin mesh acisi -> yapistir

SCALE_BOOST = 8.0
STRICT_ANGLE_MATCH = True   # True: kare bazli aci (yumusatma yok)
BANK_BLEND = True           # 5° arasi iki mesh gorunumu karistir
GLOBAL_SCALE = True

SMOOTH_WINDOW = 5           # STRICT=False ise
MAX_FRAMES = None           # test: 20
WRITE_DEBUG = True          # unity | mesh | composite (ilk N kare)
DEBUG_MAX_FRAMES = 12
MP4_FPS = 15.0
COPY_TO_DRIVE = False
# ================================

sys.path.insert(0, "/content/g3d")
from vrcaps_colab_bootstrap import bootstrap

bootstrap()
from colab_content_paths import install_sys_path

install_sys_path()

# Bolum 0 git pull sonrasi eski modul cache'ini temizle
for _mod in list(sys.modules):
    if _mod in ("gaze_composite", "export_gaze_views", "coating_utils"):
        del sys.modules[_mod]

_gaze_py = Path("/content/g3d/captures/gaze_composite.py")
if not _gaze_py.exists() or "def load_gaze_rows" not in _gaze_py.read_text(encoding="utf-8"):
    raise RuntimeError(
        "Eski gaze_composite.py — once Bolum 0 calistirin (git pull).\n"
        "Hala olmazsa: !rm -rf /content/g3d  sonra Bolum 0."
    )

from colab_content_paths import (
    DATASET_DRIVE,
    GAZE_COMPOSITE_DRIVE,
    GAZE_COMPOSITE_OUT,
    UNITY_DATASET,
    copy_tree_if_requested,
    drive_mounted,
    resolve_view_bank,
    sync_dataset_from_drive,
)
from export_gaze_views import export_gaze_views
import importlib
import gaze_composite as _gaze_composite

importlib.reload(_gaze_composite)
from gaze_composite import build_strip, composite_dataset


def ensure_dataset_on_content() -> Path:
    if (UNITY_DATASET / "rgb").is_dir():
        return UNITY_DATASET
    if drive_mounted() and (DATASET_DRIVE / "rgb").is_dir():
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        print("Drive dataset -> /content kopyalaniyor...")
        return sync_dataset_from_drive(UNITY_DATASET)
    raise FileNotFoundError(
        f"Dataset yok.\n  Bolum 5b: MyDrive/vrcaps/medical_gan_dataset\n  veya {UNITY_DATASET}"
    )


def require_view_bank(view_bank: Path) -> None:
    manifest = view_bank / "view_manifest.json"
    if not manifest.exists():
        raise FileNotFoundError(
            f"View bank yok: {view_bank}\nOnce Bolum 3 calistirin."
        )
    meta = json.loads(manifest.read_text(encoding="utf-8"))
    views = meta.get("views", [])
    azs = [float(v["azimuth_deg"]) for v in views]
    print(f"View bank: {view_bank}")
    print(f"  {len(views)} aci  ({min(azs):.0f}..{max(azs):.0f})")


def show_previews(out_dir: Path) -> None:
    from IPython.display import Image, display

    debug = sorted((out_dir / "debug").glob("match_*.png"))[:4]
    for p in debug:
        print(p.name)
        display(Image(filename=str(p), width=900))
    strip = out_dir / "composite_strip.png"
    if strip.exists():
        print("composite_strip.png")
        display(Image(filename=str(strip), width=900))


dataset = ensure_dataset_on_content()
view_bank = resolve_view_bank()
require_view_bank(view_bank)

gaze_csv = dataset / "poses" / "gaze_views.csv"
if not gaze_csv.exists():
    print("gaze_views.csv uretiliyor...")
    export_gaze_views(dataset, write_plot=False)

if GAZE_COMPOSITE_OUT.exists():
    shutil.rmtree(GAZE_COMPOSITE_OUT)
GAZE_COMPOSITE_OUT.mkdir(parents=True, exist_ok=True)

smooth_angles = not STRICT_ANGLE_MATCH
smooth_distance = not STRICT_ANGLE_MATCH

print("\n=== Aci eslestirmeli composite ===")
print("  dataset:  ", dataset)
print("  view_bank:", view_bank)
print("  strict:   ", STRICT_ANGLE_MATCH)

summary = composite_dataset(
    dataset,
    view_bank,
    GAZE_COMPOSITE_OUT,
    scale_boost=SCALE_BOOST,
    global_scale=GLOBAL_SCALE,
    smooth_window=SMOOTH_WINDOW,
    smooth_angles=smooth_angles,
    smooth_distance=smooth_distance,
    bank_blend=BANK_BLEND,
    max_frames=MAX_FRAMES,
    write_mp4=True,
    write_debug=WRITE_DEBUG,
    debug_max_frames=DEBUG_MAX_FRAMES,
    mp4_fps=MP4_FPS,
)
strip = build_strip(GAZE_COMPOSITE_OUT)
print("Kare:", summary["composite_count"])
print("Cikti:", GAZE_COMPOSITE_OUT / "rgb")
print("Debug:", GAZE_COMPOSITE_OUT / "debug")
print("MP4:", GAZE_COMPOSITE_OUT / "trajectory_composite.mp4")

copy_tree_if_requested(GAZE_COMPOSITE_OUT, GAZE_COMPOSITE_DRIVE, enabled=COPY_TO_DRIVE)
show_previews(GAZE_COMPOSITE_OUT)

